## 0)  Project Root

In [22]:
import sys
from pathlib import Path


def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()

[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [23]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Evaluating on:", device)

Evaluating on: mps


In [24]:
from pathlib import Path
import json
import torch


SEQ_DIR = PROJECT_ROOT / "03_Sequences"
RUN_DIR = PROJECT_ROOT / "runs" / "LSTMClassifier_bs16_20251219-071043_UTC"  # <- pick one


CONFIG_PATH = RUN_DIR / "config.json"
CKPT_PATH   = RUN_DIR / "best_model.pt"


In [25]:
from utils.data_utils import make_test_loader

test_loader, X_test_raw, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

model_class_name = config["model_class"]
model_kwargs     = config["model_kwargs"]

In [26]:
from models import LSTMClassifier  # add more as you create them

MODEL_REGISTRY = {
    "LSTMClassifier": LSTMClassifier,
    # "GRUClassifier": GRUClassifier,
    # "TransformerClassifier": TransformerClassifier,
}

ModelClass = MODEL_REGISTRY[model_class_name]
model = ModelClass(**model_kwargs).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()


LSTMClassifier(
  (lstm): LSTM(15, 64, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [27]:
import numpy as np
from tqdm import tqdm

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc="Predicting (test)", leave=False):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1)

        logits = model(X_batch).view(-1)                 # ensure shape (batch,)
        probs  = torch.sigmoid(logits)                   # UP probability in [0,1]
        preds  = (probs >= 0.5).long()

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_probs  = np.array(all_probs).reshape(-1)
all_preds  = np.array(all_preds).reshape(-1).astype(int)
all_labels = np.array(all_labels).reshape(-1).astype(int)


In [28]:
meta = json.loads((SEQ_DIR / "meta.json").read_text())

t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
t_to_end_min_values = X_test_raw[:, -1, t_to_end_min_idx].astype(int)

assert len(t_to_end_min_values) == len(all_labels) == len(all_probs)


In [29]:
import pandas as pd

df = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "p_up": all_probs,
    "y_pred": all_preds,
})


In [35]:
import numpy as np
import json

df["tp"] = ((df.y_true==1) & (df.y_pred==1)).astype(int)
df["fp"] = ((df.y_true==0) & (df.y_pred==1)).astype(int)
df["tn"] = ((df.y_true==0) & (df.y_pred==0)).astype(int)
df["fn"] = ((df.y_true==1) & (df.y_pred==0)).astype(int)
df["correct"] = (df.y_true == df.y_pred).astype(int)

stats = df.groupby("t_to_end_min").agg(
    count=("correct","count"),
    tp=("tp","sum"),
    fp=("fp","sum"),
    tn=("tn","sum"),
    fn=("fn","sum"),
    accuracy=("correct","mean"),
    avg_p_up=("p_up","mean"),
    base_rate=("y_true","mean"),
).reset_index()

# -----------------------------
# Add all classification metrics
# -----------------------------
def safe_div(num, den):
    return np.where(den == 0, np.nan, num / den)

# Positive-class metrics (UP = 1)
stats["precision"] = safe_div(stats["tp"], stats["tp"] + stats["fp"])   # PPV
stats["recall"]    = safe_div(stats["tp"], stats["tp"] + stats["fn"])   # TPR / Sensitivity

stats["f1"] = safe_div(
    2 * stats["precision"] * stats["recall"],
    stats["precision"] + stats["recall"]
)

# Negative-class metrics (DOWN = 0)
stats["specificity"] = safe_div(stats["tn"], stats["tn"] + stats["fp"]) # TNR
stats["npv"]         = safe_div(stats["tn"], stats["tn"] + stats["fn"]) # Negative Predictive Value

# Optional but common:
stats["fpr"] = safe_div(stats["fp"], stats["fp"] + stats["tn"])         # False Positive Rate
stats["fnr"] = safe_div(stats["fn"], stats["fn"] + stats["tp"])         # False Negative Rate

# Percent formats (optional)
stats["accuracy_pct"] = stats["accuracy"] * 100
stats["precision_pct"] = stats["precision"] * 100
stats["recall_pct"] = stats["recall"] * 100
stats["f1_pct"] = stats["f1"] * 100
stats["specificity_pct"] = stats["specificity"] * 100
stats["npv_pct"] = stats["npv"] * 100

# -----------------------------
# Save to JSON
# -----------------------------
stats_json = stats.to_dict(orient="records")

OUT_DIR = RUN_DIR / "eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "per_t_to_end_min_stats.json"
out_path.write_text(json.dumps(stats_json, indent=2))

print(stats.to_string(index=False))
print(f"\nSaved per-minute stats to {out_path}")


 t_to_end_min  count    tp   fp    tn   fn  accuracy  avg_p_up  base_rate  precision   recall       f1  specificity      npv      fpr      fnr  accuracy_pct  precision_pct  recall_pct    f1_pct  specificity_pct   npv_pct
            1  31564 15206  138 15685  535  0.978678  0.470371   0.498701   0.991006 0.966012 0.978350     0.991279 0.967016 0.008721 0.033988     97.867824      99.100626   96.601232 97.834969        99.127852 96.701603
            2  31564 14568  851 14972 1173  0.935876  0.469916   0.498701   0.944808 0.925481 0.935045     0.946218 0.927346 0.053782 0.074519     93.587631      94.480835   92.548123 93.504493        94.621753 92.734593
            3  31564 14111 1444 14379 1630  0.902611  0.471042   0.498701   0.907168 0.896449 0.901777     0.908740 0.898182 0.091260 0.103551     90.261057      90.716811   89.644876 90.177658        90.874044 89.818227
            4  31564 13696 1920 13903 2045  0.874382  0.471372   0.498701   0.877049 0.870084 0.873553     0.878658 

## Calibration & Threshold

In [47]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calibration_curve(df_sub, bins=10, min_count=5):
    df_sub = df_sub.copy()
    df_sub["prob_bin"] = pd.cut(df_sub["p_up"], bins=bins)

    calib = df_sub.groupby("prob_bin").agg(
        avg_p=("p_up", "mean"),
        freq_up=("y_true", "mean"),
        count=("y_true", "count"),
    ).dropna()

    return calib[calib["count"] >= min_count]


# ----------------------------
# Calibration plots per t
# ----------------------------
OUT_DIR = RUN_DIR / "eval" / "calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for t, g in df.groupby("t_to_end_min"):
    calib = calibration_curve(g, bins=10, min_count=5)
    if len(calib) < 2:
        continue

    total_n = len(g)

    plt.figure()

    # Model calibration curve
    plt.plot(
        calib["avg_p"],
        calib["freq_up"],
        marker="o",
        label="Model calibration"
    )

    # Perfect calibration reference
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="green",
        label="Perfect calibration"
    )

    # Annotate each point with absolute + percentage
    for _, row in calib.iterrows():
        pct = 100 * row["count"] / total_n
        plt.annotate(
            f"n={int(row['count'])}\n({pct:.1f}%)",
            (row["avg_p"], row["freq_up"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
        )

    # Axis ticks at 0.1
    ticks = np.linspace(0, 1, 11)
    plt.xticks(ticks)
    plt.yticks(ticks)
    plt.grid(True, which="both", linestyle="--", alpha=0.5)

    plt.xlabel("Average predicted probability")
    plt.ylabel("Empirical UP frequency")
    plt.title(f"Calibration curve (t_to_end_min={t}, n={total_n})")
    plt.legend(loc="lower right")

    plt.savefig(
        OUT_DIR / f"calibration_t{t}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()

# ------------------------------------
# 2) Threshold stats + plots per t
# ------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

THRESHOLDS = np.linspace(0.5, 0.9, 9)

BASE_DIR = RUN_DIR / "eval" / "threshold_analysis"
UP_DIR   = BASE_DIR / "up"
DN_DIR   = BASE_DIR / "down"
BASE_DIR.mkdir(parents=True, exist_ok=True)
UP_DIR.mkdir(parents=True, exist_ok=True)
DN_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Build threshold tables
# --------------------------
rows_up = []
rows_dn = []

for t, g in df.groupby("t_to_end_min"):
    n_total = len(g)

    for tau in THRESHOLDS:
        # ---- UP: confident UP when p_up >= tau
        sel_up = g[g["p_up"] >= tau]
        if len(sel_up) > 0:
            rows_up.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_up)),
                "coverage": float(len(sel_up) / n_total),
                "accuracy": float((sel_up["y_pred"] == sel_up["y_true"]).mean()),
                "base_rate_up": float(sel_up["y_true"].mean()),  # UP rate in selected set
            })

        # ---- DOWN: confident DOWN when p_up <= (1 - tau)
        sel_dn = g[g["p_up"] <= (1 - tau)]
        if len(sel_dn) > 0:
            # if we "act DOWN", predicted label is 0
            rows_dn.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_dn)),
                "coverage": float(len(sel_dn) / n_total),
                "accuracy": float((sel_dn["y_true"] == 0).mean()),
                "base_rate_up": float(sel_dn["y_true"].mean()),  # should be low if DOWN is correct
            })

thr_up = pd.DataFrame(rows_up)
thr_dn = pd.DataFrame(rows_dn)

# --------------------------
# Plot helper (no extra funcs)
# --------------------------
for side, thr_df, out_dir in [
    ("UP", thr_up, UP_DIR),
    ("DOWN", thr_dn, DN_DIR),
]:
    if thr_df.empty:
        continue

    for t, g in thr_df.groupby("t_to_end_min"):
        g = g.sort_values("threshold")

        plt.figure()
        ax = plt.gca()

        # Accuracy (blue)
        line1, = ax.plot(
            g["threshold"],
            g["accuracy"],
            marker="o",
            color="blue",
            label="Accuracy given predicted probability ≥ τ"
        )

        # X axis label depends on side
        if side == "UP":
            ax.set_xlabel("Probability threshold (p_up ≥ τ)")
        else:
            ax.set_xlabel("Probability threshold (p_up ≤ 1 − τ)")

        ax.set_ylabel("Accuracy given predicted probability ≥ τ")

        # Coverage (green)
        ax2 = ax.twinx()
        line2, = ax2.plot(
            g["threshold"],
            g["coverage"] * 100,
            marker="s",
            linestyle="--",
            color="green",
            label="Coverage (%)"
        )
        ax2.set_ylabel("Coverage (%)")

        # Annotate absolute counts
        for _, row in g.iterrows():
            ax.annotate(
                f"{int(row['n_samples'])}",
                (row["threshold"], row["accuracy"]),
                textcoords="offset points",
                xytext=(0, 6),
                ha="center",
                fontsize=8,
            )

        ax.grid(True)
        plt.title(f"{side}: Accuracy & coverage vs threshold (t_to_end_min={t})")

        ax.legend(handles=[line1, line2], loc="lower right")

        plt.savefig(out_dir / f"accuracy_coverage_t{t}.png", dpi=150, bbox_inches="tight")
        plt.close()

# --------------------------
# Save summary.json (both)
# --------------------------
summary = {
    "thresholds": [float(x) for x in THRESHOLDS],
    "up": thr_up.to_dict(orient="records"),
    "down": thr_dn.to_dict(orient="records"),
}

(BASE_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print("Saved:", BASE_DIR / "summary.json")
print("UP plots  :", UP_DIR)
print("DOWN plots:", DN_DIR)


/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34487/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34487/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34487/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the fut

Saved: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_bs16_20251219-071043_UTC/eval/threshold_analysis/summary.json
UP plots  : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_bs16_20251219-071043_UTC/eval/threshold_analysis/up
DOWN plots: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_bs16_20251219-071043_UTC/eval/threshold_analysis/down
